# 04 — Двухэтапная рекомендательная система с CatBoostRanker

В ноутбуке проверяется, улучшает ли обучение ранжированию рекомендации топ-10 по сравнению с существующими моделями коллаборативной фильтрации. Все метрики рассчитаны на временных выборках Retailrocket; результаты заранее не предполагаются.

## 1. Постановка задачи

Для каждого пользователя нужно рекомендовать до десяти ещё не просмотренных товаров, с которыми он, вероятно, взаимодействует в следующем временном интервале. Используем бинарную релевантность: любой будущий просмотр, переход в корзину или заказ получает метку 1. Метка 0 означает только, что взаимодействие не было замечено в целевом интервале.

Основная метрика — макроусреднённая NDCG@10. Recall@10, Precision@10, покрытие каталога Coverage@10 и Recall@100 кандидатов показывают дополнительные стороны качества. Тестовый период не используется до фиксации всех решений по модели.

## 2. Архитектура рекомендательной системы из двух этапов

```mermaid
flowchart LR
  H[История взаимодействий] --> A[Кандидаты от implicit ALS]
  H --> K[Кандидаты от ItemKNN]
  A --> U[Объединение и удаление повторов, до 100 от каждой модели]
  K --> U
  U --> F[Признаки пользователя, товара и кандидата на момент среза]
  F --> R[CatBoostRanker]
  R --> T[Топ-10 ещё не просмотренных товаров]
```

Ранжировщик может переставлять только товары, найденные генераторами. Поэтому полнота кандидатов задаёт верхнюю границу полноты всей системы. ALS — готовая библиотечная реализация из `implicit`; в этом ноутбуке в проекте используется готовая реализация из `implicit`.

## Временной протокол и воспроизводимый эксперимент

Исходное хронологическое разбиение 70/15/15 не меняется. Для обучения ранжировщика внутри обучающего интервала выделяется дополнительный срез по уникальным временным отметкам: первые 80% составляют доступную историю, последние 20% — целевой интервал. Модели-кандидатогенераторы и признаки для этого среза используют только предшествующую историю. Для проверочной выборки кандидаты и признаки строятся только по обучающей; для тестовой — по обучающей и проверочной. Когорта и каталог выбранного режима выбираются только по обучающей выборке. Будущие ID товаров используются только для формирования целевых меток после генерации кандидатов.

## Единый протокол и режим бенчмарка

Основной протокол во всех ноутбуках — **binary relevance**: любой новый товар с будущим событием релевантен (1), независимо от типа и повторов события. Macro NDCG@10, Recall@10 и Precision@10 считаются на одинаковых пользователях с новыми целями в фиксированном каталоге; Precision всегда делится на 10. Веса 1/3/5 остаются только сигналом обучения. Graded NDCG прежнего исследования хранится в `outputs/archive/pre_binary_unification/` и не смешивается с текущими результатами.

`BENCHMARK_MODE=dev` (по умолчанию): до 20 000 пользователей / 1 500 товаров. `full`: до 100 000 / 10 000. Это расширенный ограниченный бенчмарк, а не весь каталог. Когорта и каталог выбираются только по train. Внутренний обучающий срез ранжировщика использует собственную прошлую историю и всех подходящих пользователей, как в исходном исследовании. Полный режим пишет артефакты в `outputs/full/`, сохраняя dev. До запуска full выполните `python -m scripts.run_experiments --mode full --estimate-only`: оценка использует только train. Сетки гиперпараметров не расширяются.

Продолжение из checkpoint включается явно через `RESUME_EXPERIMENTS=1`. По умолчанию поиск выполняется заново; CLI full использует checkpoint для незавершённого этапа. Режим и пути определяются `BENCHMARK_MODE`, исходный код для dev/full одинаков.


In [ ]:
from pathlib import Path
import json
import sys
import os

import gc
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import catboost

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "src" / "retailrocket.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the project directory.")
sys.path.insert(0, str(ROOT))
from src.benchmark import (
    get_benchmark_config, artifact_root, benchmark_manifest, validate_selection,
)
CONFIG = get_benchmark_config()
USE_CHECKPOINTS = os.environ.get("RESUME_EXPERIMENTS", "0") == "1"
ARTIFACTS = artifact_root(ROOT, CONFIG)
PROCESSED = ROOT / "data" / "processed"
FIGURES = ARTIFACTS / "figures"
METRICS = ARTIFACTS / "metrics"
MODELS = ARTIFACTS / "models"
for folder in (FIGURES, METRICS, MODELS):
    folder.mkdir(parents=True, exist_ok=True)

from src.candidate_generation import (
    candidate_recall,
    generate_candidate_frame,
)
from src.evaluation import evaluate_recommender
from src.ranker import (
    CandidateRankerRecommender,
    CATEGORICAL_FEATURES,
    CF_FEATURES,
    DEFAULT_RANKER_PARAMS,
    RANKER_FEATURES,
    feature_importance_frame,
    fit_catboost_ranker,
    make_ranker_pool,
)
from src.ranking_features import (
    attach_binary_targets,
    build_ranking_features,
    split_history_target,
)
from src.recommendation import (
    ImplicitLibraryALSRecommender,
    ItemKNNRecommender,
    PopularityRecommender,
    RecentPopularityRecommender,
    build_interaction_data,
    select_benchmark_users,
    select_catalog_items,
)
from src.retailrocket import load_processed_events

MIN_HISTORY = CONFIG.min_history
MAX_USERS = CONFIG.max_users
MAX_ITEMS = CONFIG.max_items
SEED = CONFIG.seed
K = CONFIG.k
RELEVANCE_MODE = CONFIG.relevance_mode
CANDIDATES_PER_MODEL = CONFIG.candidates_per_model

train = load_processed_events(PROCESSED / "train.csv")
validation = load_processed_events(PROCESSED / "validation.csv")
cohort = select_benchmark_users(train, min_history=5, max_users=MAX_USERS,
                                random_state=SEED)
catalog = select_catalog_items(train, max_items=MAX_ITEMS)
train_benchmark = train[
    train.visitorid.isin(cohort) & train.itemid.isin(catalog)
].copy()
validation_benchmark = validation[
    validation.visitorid.isin(cohort) & validation.itemid.isin(catalog)
].copy()

assert train.event_time.max() < validation.event_time.min()
print("Версия CatBoost:", catboost.__version__)
with (METRICS / "selected_model_params.json").open() as file:
    selected_parameters = json.load(file)
manifest = benchmark_manifest(train, cohort, catalog, CONFIG)
validate_selection(selected_parameters, manifest)
itemknn_params = selected_parameters["itemknn"]["parameters"]
als_params = selected_parameters["implicit_als"]["parameters"]

print("Бенчмарк: {:,} пользователей и {:,} товаров, отобранных по обучающей выборке.".format(
    len(cohort), len(catalog)))

## 3. Генерация кандидатов — обучающий срез

Обе модели обучаются только на ранней части истории внутри обучающей выборки. Для каждого пользователя объединяются до 100 ещё не просмотренных товаров от каждой модели, повторы удаляются. Если товар предложен только одной моделью, у второй модели остаются пропущенные оценки и ранги, а также явный признак источника; пропуски не подменяются настоящим нулевым баллом.

In [ ]:
rank_history, rank_target, rank_cutoff = split_history_target(
    train, history_fraction=0.8
)
rank_user_counts = rank_history.groupby("visitorid").size()
rank_users = sorted(int(user_id) for user_id in
                    rank_user_counts[rank_user_counts >= 5].index)
rank_catalog = select_catalog_items(rank_history, max_items=MAX_ITEMS)
rank_target_benchmark = rank_target[
    rank_target.visitorid.isin(rank_users)
    & rank_target.itemid.isin(rank_catalog)
].copy()
rank_candidate_users = sorted(
    rank_target_benchmark.visitorid.astype(int).unique()
)
assert rank_history.event_time.max() < rank_cutoff
assert rank_target.event_time.min() >= rank_cutoff
assert rank_target.event_time.max() < validation.event_time.min()

rank_data = build_interaction_data(
    rank_history, user_ids=rank_users, item_ids=rank_catalog
)
rank_als = ImplicitLibraryALSRecommender(**als_params).fit(rank_data)
rank_itemknn = ItemKNNRecommender(**itemknn_params).fit(rank_data)
rank_candidates = generate_candidate_frame(
    rank_candidate_users, rank_history, rank_als, rank_itemknn,
    candidates_per_model=CANDIDATES_PER_MODEL,
)
rank_features = build_ranking_features(
    rank_history, rank_candidates, rank_cutoff, snapshot_id="rank_train"
)
rank_training = attach_binary_targets(
    rank_features, rank_target_benchmark, rank_cutoff
)
print("Граница обучающего среза:", rank_cutoff)
print("Строк-кандидатов:", len(rank_training),
      "| положительных пар-кандидатов:", int(rank_training.target.sum()),
      "| пользователей хотя бы с одним положительным кандидатом:",
      int(rank_training.groupby("user_id").target.max().sum()))
display(rank_training[["als_score", "als_rank", "has_als",
                       "itemknn_score", "itemknn_rank", "has_itemknn",
                       "candidate_source", "target"]].head().rename(columns={
                           "als_score": "Оценка ALS", "als_rank": "Ранг ALS",
                           "has_als": "Есть в ALS", "itemknn_score": "Оценка ItemKNN",
                           "itemknn_rank": "Ранг ItemKNN", "has_itemknn": "Есть в ItemKNN",
                           "candidate_source": "Источник кандидата", "target": "Целевая метка",
                       }).replace({"Источник кандидата": {
                           "both": "Обе модели", "als_only": "Только ALS",
                           "itemknn_only": "Только ItemKNN",
                       }}))
rank_snapshot_start = str(rank_history.event_time.min())
rank_snapshot_target_end = str(rank_target.event_time.max())
del rank_history, rank_target, rank_target_benchmark, rank_candidates, rank_features
del rank_data, rank_als, rank_itemknn
gc.collect()


## 4. Формирование признаков

Для каждого кандидата рассчитываются оценки и ранги моделей-источников, а также признаки наличия кандидата в их списках. Признаки пользователя включают число событий каждого типа, число уникальных товаров в истории и часы с последней активности. Признаки товара включают число событий каждого типа, число уникальных пользователей и количество событий за предыдущие семь дней. Функция построения признаков отклоняет историю с событиями во время среза или после него, поэтому будущие взаимодействия не попадут в агрегаты.

In [ ]:
feature_columns = RANKER_FEATURES
FEATURE_LABELS = {
    "als_score": "Оценка ALS", "als_rank": "Ранг ALS",
    "has_als": "Есть в ALS", "itemknn_score": "Оценка ItemKNN",
    "itemknn_rank": "Ранг ItemKNN", "has_itemknn": "Есть в ItemKNN",
    "candidate_source": "Источник кандидата",
    "user_view_count": "Просмотры пользователя",
    "user_addtocart_count": "Добавления в корзину пользователя",
    "user_transaction_count": "Покупки пользователя",
    "user_unique_items": "Уникальные товары в истории пользователя",
    "user_last_activity_hours": "Часы с последней активности пользователя",
    "user_total_events": "Всего событий пользователя",
    "item_view_count": "Просмотры товара",
    "item_addtocart_count": "Добавления товара в корзину",
    "item_transaction_count": "Покупки товара",
    "item_unique_users": "Уникальные пользователи товара",
    "item_event_count_7d": "События товара за 7 дней",
    "target": "Целевая метка",
}
feature_summary = rank_training[feature_columns + ["target"]].describe(include="all").T
feature_summary = feature_summary.rename(columns={
    "count": "Количество", "unique": "Уникальных значений",
    "top": "Самое частое", "freq": "Частота",
    "mean": "Среднее", "std": "Стандартное отклонение",
    "min": "Минимум", "max": "Максимум",
    "25%": "25-й процентиль", "50%": "Медиана", "75%": "75-й процентиль",
})
feature_summary.index = feature_summary.index.map(
    lambda feature: FEATURE_LABELS.get(feature, feature)
)
display(feature_summary)

## 5. Формирование целевой переменной

Кандидат получает метку 1, если пара пользователь–товар встретилась в будущем целевом интервале, независимо от типа события; иначе метка равна 0. Повторные события одной пары остаются одной положительной меткой. Целевой интервал обучающего среза находится внутри исходной обучающей выборки. Метки проверочной и тестовой выборок присоединяются только после того, как кандидаты и признаки построены по доступной к тому моменту истории.

## 6. Обучение и небольшой поиск CatBoostRanker

Сравним три конфигурации с одинаковым seed, максимумом 500 итераций и ранней остановкой по NDCG@10 на проверочной выборке: исходную (глубина 6, шаг обучения 0,05), более неглубокую (глубина 4, 0,05) и с меньшим шагом (глубина 6, 0,03). Выбирается модель с полным набором признаков; три дополнительных варианта ниже нужны для абляции на проверочной выборке.

**Гипотеза.** Уменьшение глубины 6→4 при шаге 0.05 снизит переобучение; уменьшение learning_rate 0.05→0.03 при глубине 6 позволит точнее оптимизировать порядок. Это два контролируемых сравнения с исходной точкой, а не полная сетка: взаимодействие глубины 4 с шагом 0.03 не проверяется. Ранняя остановка выбирает число деревьев на той же validation, поэтому результат подбора оптимистичен.

In [ ]:
train_users_for_eval = sorted(validation_benchmark.visitorid.astype(int).unique())
validation_data = build_interaction_data(
    train_benchmark, user_ids=cohort, item_ids=catalog
)
validation_als = ImplicitLibraryALSRecommender(**als_params).fit(validation_data)
validation_itemknn = ItemKNNRecommender(**itemknn_params).fit(validation_data)
validation_candidates = generate_candidate_frame(
    train_users_for_eval, train_benchmark, validation_als, validation_itemknn,
    candidates_per_model=CANDIDATES_PER_MODEL,
)
validation_cutoff = validation.event_time.min()
validation_features = build_ranking_features(
    train, validation_candidates, validation_cutoff, snapshot_id="validation"
)
validation_rank_data = attach_binary_targets(
    validation_features, validation, validation_cutoff
)

catboost_search = [
    {"name": "Глубина 6, шаг 0,05 (исходная)", "depth": 6, "learning_rate": 0.05},
    {"name": "Глубина 4, шаг 0,05", "depth": 4, "learning_rate": 0.05},
    {"name": "Глубина 6, шаг 0,03", "depth": 6, "learning_rate": 0.03},
]
cached_tuning_path = METRICS / "ltr_catboost_tuning_validation.csv"
cached_tuning = pd.read_csv(cached_tuning_path) if USE_CHECKPOINTS and cached_tuning_path.exists() else None
configs_to_fit = catboost_search
if cached_tuning is not None:
    frozen_name = cached_tuning.loc[cached_tuning["ndcg@10"].idxmax(), "name"]
    configs_to_fit = [config for config in catboost_search if config["name"] == frozen_name]
    print("Восстановление только выбранной модели; поиск загружен из checkpoint:", frozen_name)
catboost_models = {}
catboost_rows = []
for config in configs_to_fit:
    params = dict(DEFAULT_RANKER_PARAMS)
    params.update({
        "depth": config["depth"],
        "learning_rate": config["learning_rate"],
        "allow_writing_files": False,
        "verbose": False,
        "thread_count": 4,
    })
    model = fit_catboost_ranker(
        rank_training, validation_rank_data, params=params,
        early_stopping_rounds=50,
    )
    catboost_models[config["name"]] = model
    validation_recommender = CandidateRankerRecommender(model, validation_features, snapshot_id="validation")
    metrics, _, _ = evaluate_recommender(
        validation_recommender, validation, train, cohort, catalog, k=K,
        relevance_mode=RELEVANCE_MODE,
    )
    catboost_rows.append({
        **config,
        "best_iteration": int(model.get_best_iteration()),
        **metrics,
    })

if cached_tuning is not None:
    recovered = catboost_rows[0]
    expected = cached_tuning.set_index("name").loc[recovered["name"]]
    np.testing.assert_allclose([recovered[m] for m in ["ndcg@10", "recall@10", "precision@10", "coverage@10"]],
                               expected[["ndcg@10", "recall@10", "precision@10", "coverage@10"]].to_numpy(dtype=float),
                               rtol=1e-7, atol=1e-9)
    assert recovered["best_iteration"] == expected["best_iteration"]
    catboost_tuning = cached_tuning
else:
    catboost_tuning = pd.DataFrame(catboost_rows)
catboost_tuning.to_csv(METRICS / "ltr_catboost_tuning_validation.csv", index=False)
best_catboost_index = int(catboost_tuning["ndcg@10"].idxmax())
best_catboost_name = catboost_tuning.loc[best_catboost_index, "name"]
ranker = catboost_models[best_catboost_name]
best_iteration = ranker.get_best_iteration()
ranker_params = dict(DEFAULT_RANKER_PARAMS)
ranker_params.update({
    "depth": int(catboost_tuning.loc[best_catboost_index, "depth"]),
    "learning_rate": float(catboost_tuning.loc[best_catboost_index, "learning_rate"]),
    "allow_writing_files": False,
    "verbose": False,
    "thread_count": 4,
})

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(catboost_tuning["name"], catboost_tuning["ndcg@10"], color="#4C78A8")
ax.set(title="CatBoostRanker: NDCG@10 на проверочной выборке", ylabel="NDCG@10", xlabel="Конфигурация")
ax.tick_params(axis="x", labelrotation=15)
for position, value in enumerate(catboost_tuning["ndcg@10"]):
    ax.text(position, value, "{:.4f}".format(value), ha="center", va="bottom")
fig.tight_layout()
fig.savefig(FIGURES / "ltr_catboost_tuning_validation.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close("all")
plt.close(fig)
catboost_tuning_display = catboost_tuning[["name", "depth", "learning_rate", "best_iteration", "ndcg@10", "recall@10"]].rename(columns={
    "name": "Конфигурация", "depth": "Глубина", "learning_rate": "Шаг обучения",
    "best_iteration": "Лучшая итерация", "ndcg@10": "NDCG@10", "recall@10": "Recall@10",
})
display(catboost_tuning_display.round(4))
print("Выбрана CatBoost-конфигурация по NDCG@10 на проверочной выборке:", best_catboost_name,
      "— {:.4f}".format(catboost_tuning.loc[best_catboost_index, "ndcg@10"]))


ranker.save_model(str(MODELS / "catboost_ranker_validation_checkpoint.cbm"))
del catboost_models
gc.collect()


In [ ]:
cb = catboost_tuning.set_index(["depth", "learning_rate"])
base = cb.loc[(6, .05), "ndcg@10"]
shallow = cb.loc[(4, .05), "ndcg@10"]
slow = cb.loc[(6, .03), "ndcg@10"]
display(Markdown(
    f"**Вывод.** При шаге 0.05 глубина 6→4: {base:.5f}→{shallow:.5f}, Δ={shallow-base:+.5f}. "
    f"При глубине 6 шаг 0.05→0.03: {base:.5f}→{slow:.5f}, Δ={slow-base:+.5f}. "
    "Это два контролируемых сравнения: эффект глубины при 0.03 не проверен. "
    "Число деревьев выбиралось ранней остановкой отдельно, поэтому эффект learning_rate включает изменение длительности обучения. "
    "NDCG оценивает порядок; Precision/Recall в таблице проверяют, сопровождается ли он увеличением попаданий. "
    "Один seed и небольшая оцениваемая когорта не подтверждают универсальность лучшей точки."
))


## Абляция признаков

**Гипотеза.** Числовые оценки генераторов и признаки популярности/активности несут дополнительный сигнал поверх рангов и источника кандидатов. Удаляем каждую группу при неизменных кандидатах и выбранных depth/learning_rate. Число деревьев каждый раз выбирает early stopping; измеряем эффект всей процедуры обучения. Полный набор фиксирован заранее для test; абляция — диагностика.

Добавляем полное удаление семи признаков генераторов: scores, ranks, has-флаги и `candidate_source`. Остаются только 11 агрегатов пользователя/товара. Кандидаты по-прежнему получены ALS/ItemKNN: этот эксперимент оценивает вклад их **признаков в перестановку фиксированных кандидатов**, а не вклад самих генераторов. Полный набор выбран до test независимо от абляции.

In [ ]:
cf_score_features = {"als_score", "itemknn_score"}
popularity_activity_features = {
    feature for feature in RANKER_FEATURES
    if feature.startswith(("user_", "item_"))
}
feature_variants = {
    "Все признаки": list(RANKER_FEATURES),
    "Без оценок ALS/ItemKNN": [
        feature for feature in RANKER_FEATURES if feature not in cf_score_features
    ],
    "Без всех признаков ALS/ItemKNN": [
        feature for feature in RANKER_FEATURES if feature not in CF_FEATURES
    ],
    "Без популярности/активности": [
        feature for feature in RANKER_FEATURES
        if feature not in popularity_activity_features
    ],
}
if USE_CHECKPOINTS and (METRICS / "ltr_ablation_validation.csv").exists():
    print("Абляции загружены из checkpoint без повторного обучения.")
    ablation_results = pd.read_csv(METRICS / "ltr_ablation_validation.csv")
    cached_users = pd.read_csv(METRICS / "ltr_ablation_per_user.csv")
    ablation_per_user = {variant: frame.drop(columns="variant").reset_index(drop=True)
                         for variant, frame in cached_users.groupby("variant", sort=False)}
    ranker_ablation_models = {}
else:
    ranker_ablation_models = {"Все признаки": ranker}
    for variant_name, features in feature_variants.items():
        if variant_name == "Все признаки":
            continue
        categorical_features = [
            feature for feature in CATEGORICAL_FEATURES if feature in features
        ]
        ranker_ablation_models[variant_name] = fit_catboost_ranker(
            rank_training,
            validation_rank_data,
            params=ranker_params,
            early_stopping_rounds=50,
            feature_columns=features,
            categorical_features=categorical_features,
        )

    ablation_rows = []
    ablation_per_user = {}
    for variant_name, features in feature_variants.items():
        model = ranker_ablation_models[variant_name]
        recommender = CandidateRankerRecommender(
            model, validation_features, feature_columns=features, snapshot_id="validation"
        )
        metrics, per_user, _ = evaluate_recommender(
            recommender, validation, train, cohort, catalog, k=K,
            relevance_mode=RELEVANCE_MODE,
        )
        ablation_per_user[variant_name] = per_user
        ablation_rows.append({
            "variant": variant_name,
            "feature_count": len(features),
            "best_iteration": int(model.get_best_iteration()),
            **metrics,
        })
    ablation_results = pd.DataFrame(ablation_rows)
    ablation_results.to_csv(METRICS / "ltr_ablation_validation.csv", index=False)
ablation_display = ablation_results[["variant", "feature_count", "best_iteration", "ndcg@10", "recall@10", "precision@10"]].rename(columns={
    "variant": "Вариант", "feature_count": "Число признаков",
    "best_iteration": "Лучшая итерация", "ndcg@10": "NDCG@10",
    "recall@10": "Recall@10", "precision@10": "Precision@10",
})
display(ablation_display.round(4))
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(ablation_results["variant"], ablation_results["ndcg@10"], color="#F58518")
ax.set(title="CatBoostRanker: абляция признаков на проверочной выборке", ylabel="NDCG@10", xlabel="Набор признаков")
ax.tick_params(axis="x", labelrotation=18)
fig.tight_layout()
fig.savefig(FIGURES / "ltr_feature_ablation_validation.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close("all")
plt.close(fig)
print("Абляция — диагностическое сравнение только на проверочной выборке; выбранная модель использует полный набор признаков.")
from src.experiment_analysis import paired_metric_comparison
ablation_intervals = paired_metric_comparison(ablation_per_user, reference="Все признаки", seed=SEED)
ablation_intervals.to_csv(METRICS / "ltr_ablation_paired_intervals.csv", index=False)
display(ablation_intervals.round(6))
pd.concat([frame.assign(variant=name) for name, frame in ablation_per_user.items()], ignore_index=True).to_csv(METRICS / "ltr_ablation_per_user.csv", index=False)

del ranker_ablation_models, validation_rank_data
gc.collect()


In [ ]:
full_ndcg = ablation_results.set_index("variant").loc["Все признаки", "ndcg@10"]
for row in ablation_intervals.itertuples():
    direction = "снижение" if row.delta < 0 else "рост"
    interval_note = "знак не установлен: CI включает ноль" if row.ci_low <= 0 <= row.ci_high else "CI исключает ноль на этой validation-когорте"
    display(Markdown(
        f"**{row.model}:** NDCG@10={full_ndcg+row.delta:.5f}, {direction} относительно полного набора "
        f"Δ={row.delta:+.5f} ({row.relative_delta_pct:+.1f}%), 95% CI [{row.ci_low:+.5f}; {row.ci_high:+.5f}]; {interval_note}."
    ))
display(Markdown(
    "**Вывод.** Разница без всех признаков генераторов измеряет их добавочный сигнал сверх агрегатов на тех же кандидатах. "
    "Сравнение с удалением только scores показывает, насколько вывод зависит от сохранённых рангов и источников. "
    "Интервалы диагностические: модель и early stopping уже выбраны по той же validation. "
    "Удаление агрегатов может изменить Coverage и Recall иначе, чем NDCG; эти компромиссы видны в таблице."
))


## 7. Сравнение систем на validation и закрытом test

**Гипотеза.** Ранжировщик улучшит порядок кандидатов относительно каждого генератора и сохранит преимущество на следующем временном интервале. Фиксируем параметры и полный набор признаков до чтения test. Все ноутбуки используют одинаковую binary relevance и параметры, выбранные по binary validation. Общие четыре модели дополнительно проверяются на совпадение метрик с 02/03. Test используется только для итогового измерения.

In [ ]:
validation_ranker = CandidateRankerRecommender(ranker, validation_features, snapshot_id="validation")
validation_models = {
    "Global Popularity": PopularityRecommender().fit(
        train_benchmark, catalog_items=catalog
    ),
    "Recent Popularity (14d)": RecentPopularityRecommender().fit(
        train_benchmark, as_of=validation_cutoff, lookback_days=14,
        catalog_items=catalog
    ),
    "ItemKNN": validation_itemknn,
    "implicit ALS": validation_als,
    "CatBoostRanker (ALS + ItemKNN)": validation_ranker,
}
validation_summaries = []
validation_per_user = {}
for name, model in validation_models.items():
    summary, per_user, _ = evaluate_recommender(
        model, validation, train, cohort, catalog, k=K,
        relevance_mode=RELEVANCE_MODE,
    )
    summary["model"] = name
    validation_summaries.append(summary)
    validation_per_user[name] = per_user
validation_results = pd.DataFrame(validation_summaries).set_index("model")
validation_results = validation_results.sort_values("ndcg@10", ascending=False)
validation_results.to_csv(METRICS / "ltr_validation_metrics.csv")
display(validation_results.rename_axis("Модель").round(4))

# Release validation model copies before allocating the final test snapshot.
del validation_models, validation_als, validation_itemknn, validation_data
del validation_ranker, validation_features, train_benchmark, validation_benchmark, model
gc.collect()
# Load the held-out period only after all model/configuration choices are fixed.
test = load_processed_events(PROCESSED / "test.csv")
assert validation.event_time.max() < test.event_time.min()
pretest_history = pd.concat([train, validation], ignore_index=True)
pretest_benchmark = pretest_history[
    pretest_history.visitorid.isin(cohort)
    & pretest_history.itemid.isin(catalog)
].copy()
test_benchmark = test[
    test.visitorid.isin(cohort) & test.itemid.isin(catalog)
].copy()
test_data = build_interaction_data(
    pretest_benchmark, user_ids=cohort, item_ids=catalog
)
test_als = ImplicitLibraryALSRecommender(**als_params).fit(test_data)
test_itemknn = ItemKNNRecommender(**itemknn_params).fit(test_data)
test_users_for_eval = sorted(test_benchmark.visitorid.astype(int).unique())
test_candidates = generate_candidate_frame(
    test_users_for_eval, pretest_benchmark, test_als, test_itemknn,
    candidates_per_model=CANDIDATES_PER_MODEL,
)
test_cutoff = test.event_time.min()
test_features = build_ranking_features(
    pretest_history, test_candidates, test_cutoff, snapshot_id="test"
)
test_ranker = CandidateRankerRecommender(ranker, test_features, snapshot_id="test")
test_models = {
    "Global Popularity": PopularityRecommender().fit(
        pretest_benchmark, catalog_items=catalog
    ),
    "Recent Popularity (14d)": RecentPopularityRecommender().fit(
        pretest_benchmark, as_of=test_cutoff, lookback_days=14,
        catalog_items=catalog
    ),
    "ItemKNN": test_itemknn,
    "implicit ALS": test_als,
    "CatBoostRanker (ALS + ItemKNN)": test_ranker,
}
test_summaries = []
test_per_user = {}
test_recommendations = {}
for name, model in test_models.items():
    summary, per_user, recommendations = evaluate_recommender(
        model, test, pretest_history, cohort, catalog, k=K,
        relevance_mode=RELEVANCE_MODE,
    )
    summary["model"] = name
    test_summaries.append(summary)
    test_per_user[name] = per_user
    test_recommendations[name] = recommendations
test_results = pd.DataFrame(test_summaries).set_index("model")
test_results = test_results.sort_values("ndcg@10", ascending=False)
test_results.to_csv(METRICS / "ltr_test_metrics.csv")
display(test_results.rename_axis("Модель").round(4))
pd.concat([frame.assign(model=name) for name, frame in test_per_user.items()], ignore_index=True).to_csv(METRICS / "ltr_test_per_user.csv", index=False)
pd.concat([frame.assign(model=name) for name, frame in validation_per_user.items()], ignore_index=True).to_csv(METRICS / "ltr_validation_per_user.csv", index=False)
# All four shared models must reproduce the primary protocol from 02 and 03.
shared = ["Global Popularity", "Recent Popularity (14d)", "ItemKNN", "implicit ALS"]
metrics_to_check = ["ndcg@10", "recall@10", "precision@10", "coverage@10", "evaluated_users"]
for split, current in [("validation", validation_results), ("test", test_results)]:
    previous = pd.read_csv(METRICS / f"model_{split}_metrics.csv", index_col="model")
    np.testing.assert_allclose(current.loc[shared, metrics_to_check], previous.loc[shared, metrics_to_check], rtol=1e-7, atol=1e-9)
print("Сопоставимость общих моделей между 02/03 и 04 подтверждена.")


In [ ]:
ranker_name = "CatBoostRanker (ALS + ItemKNN)"
r = test_results.loc[ranker_name, "ndcg@10"]
for name in ["ItemKNN", "implicit ALS"]:
    b = test_results.loc[name, "ndcg@10"]
    display(Markdown(f"**Test, CatBoost против {name}:** {r:.5f} против {b:.5f}; Δ={r-b:+.5f}, {100*(r/b-1):+.1f}%."))
display(Markdown("Это наблюдаемые средние, а не доказанная устойчивость. Coverage оцениваем отдельно; знак парной разницы проверяем ниже на той же когорте."))


In [ ]:
from src.experiment_analysis import paired_metric_comparison
test_intervals = paired_metric_comparison(test_per_user, reference="CatBoostRanker (ALS + ItemKNN)", seed=SEED)
display(test_intervals.round(6))
test_intervals.to_csv(METRICS / "ltr_test_paired_intervals.csv", index=False)
fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(test_intervals["delta"], test_intervals["model"], xerr=[test_intervals["delta"]-test_intervals["ci_low"], test_intervals["ci_high"]-test_intervals["delta"]], fmt="o")
ax.axvline(0, color="gray", linestyle="--")
ax.set(xlabel="NDCG@10 модели минус CatBoost; 95% bootstrap CI", title="Неопределённость итогового сравнения")
plt.show()
plt.close("all")

Отрицательная разница означает преимущество CatBoost; если весь интервал ниже нуля, данные поддерживают этот знак на текущей когорте. Если интервал включает ноль, наблюдаемый прирост не позволяет уверенно установить знак. Это парный bootstrap пользователей с фиксированными обученными моделями, а не проверка устойчивости к seed, временным срезам или зависимости пользователей через общие товары. По этим интервалам параметры повторно не подбираем.



In [ ]:
for row in test_intervals.itertuples():
    verdict = "знак не установлен (ноль внутри CI)" if row.ci_low <= 0 <= row.ci_high else "CI исключает ноль на этой когорте"
    display(Markdown(f"**{row.model} минус CatBoost:** Δ={row.delta:+.5f}, 95% CI [{row.ci_low:+.5f}; {row.ci_high:+.5f}]; {verdict}."))
display(Markdown("Проверяем практическую величину прироста отдельно от исключения нуля. Интервалы не скорректированы за множественные сравнения; повторный подбор по test не проводится."))


## Полнота кандидатов

**Гипотеза.** Объединение ALS и ItemKNN найдёт больше будущих целей, чем каждый генератор отдельно. Таблица и график проверяют, ограничен ли ранжировщик уже на первом этапе.

In [ ]:
validation_candidate_recall = candidate_recall(
    validation_candidates, validation, train, cohort, catalog,
    k=CANDIDATES_PER_MODEL,
)
validation_candidate_recall["split"] = "validation"
test_candidate_recall = candidate_recall(
    test_candidates, test, pretest_history, cohort, catalog,
    k=CANDIDATES_PER_MODEL,
)
test_candidate_recall["split"] = "test"
candidate_recall_results = pd.concat(
    [validation_candidate_recall, test_candidate_recall], ignore_index=True
)
candidate_recall_results.to_csv(METRICS / "ltr_candidate_recall.csv", index=False)
candidate_recall_display = candidate_recall_results.copy()
candidate_recall_display["split"] = candidate_recall_display["split"].map(
    {"validation": "Проверочная выборка", "test": "Тестовая выборка"}
)
display(candidate_recall_display.rename(columns={
    "split": "Выборка", "generator": "Генератор кандидатов",
    "metric": "Метрика", "cutoff": "Бюджет кандидатов", "recall": "Полнота", "users": "Число пользователей",
}))


In [ ]:
candidate_recall_results.assign(label=lambda f: f["generator"] + " — " + f["metric"]).pivot(index="label", columns="split", values="recall").plot.barh(figsize=(10, 4))
plt.xlabel("Macro Recall: 100 на источник, до 200 в объединении")
plt.tight_layout()
plt.savefig(FIGURES / "ltr_candidate_recall.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close("all")


In [ ]:
for split, frame in candidate_recall_results.groupby("split"):
    union = frame.loc[frame["cutoff"] == 2*CANDIDATES_PER_MODEL, "recall"].iloc[0]
    source_best = frame.loc[frame["cutoff"] == CANDIDATES_PER_MODEL, "recall"].max()
    display(Markdown(f"**{split}:** объединение Recall@200={union:.4f}, лучший отдельный Recall@100={source_best:.4f}; "
                     f"в среднем {1-union:.1%} новых целей не достигают ранжировщика. "
                     "Прирост объединения включает удвоенный бюджет; это не сравнение при равной длине списка. "
                     "Полнота ограничивает Recall, но не задаёт численный потолок NDCG."))


## 8. Важность признаков и сравнение моделей

Важность признаков описывает вклад в работу модели, но не доказывает причинно-следственную связь. Она рассчитана для обученного ранжировщика и показана вместе с метриками на проверочной и тестовой выборках, полученными по единому протоколу.

**Гипотеза.** Популярность/активность и сигналы генераторов входят в наиболее используемые признаки. Проверяем важность описательно; доказательством полезности групп служит абляция выше, а не сама важность.

In [ ]:
metric_names = ["ndcg@10", "recall@10", "precision@10", "coverage@10"]
comparison = pd.concat(
    {"validation": validation_results, "test": test_results},
    names=["split"],
).reset_index()
comparison["split"] = comparison["split"].map(
    {"validation": "Проверочная выборка", "test": "Тестовая выборка"}
)
fig, axes = plt.subplots(2, 2, figsize=(18, 7))
for axis, metric in zip(axes.flat, metric_names):
    pivot = comparison.pivot(index="model", columns="split", values=metric)
    pivot.plot(kind="bar", ax=axis, color=["#4C78A8", "#F58518"])
    axis.set_title(metric.upper())
    axis.set_xlabel("")
    axis.tick_params(axis="x", labelrotation=55)
    axis.legend(title="Выборка", fontsize=8)
    axis.grid(axis="y", alpha=0.25)
fig.suptitle("Бинарная релевантность: сравнение моделей")
fig.tight_layout()
fig.savefig(FIGURES / "learning_to_rank_metrics.png", dpi=300,
            bbox_inches="tight")
plt.show()
plt.close("all")

importance_train = rank_training[
    rank_training.groupby(["snapshot_id", "user_id"])["target"].transform("max") > 0
]
importance = feature_importance_frame(ranker, make_ranker_pool(importance_train))
importance.to_csv(METRICS / "ltr_feature_importance.csv", index=False)
importance_display = importance.head(15).copy()
importance_display["feature"] = importance_display["feature"].map(
    lambda feature: FEATURE_LABELS.get(feature, feature)
)
display(importance_display.rename(columns={"feature": "Признак", "importance": "Важность"}))
fig, ax = plt.subplots(figsize=(8, 6))
top_importance = importance.head(15).sort_values("importance")
ax.barh(top_importance["feature"], top_importance["importance"],
        color="#4C78A8")
ax.set(title="Важность признаков CatBoost", xlabel="Важность", ylabel="")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "ltr_feature_importance.png", dpi=300,
            bbox_inches="tight")
plt.show()
plt.close("all")

In [ ]:
display(Markdown("**Вывод по важности.** Первые пять признаков текущей модели: " + ", ".join(importance.head(5)["feature"]) + ". Важность описательная; причинный вклад отдельного признака не установлен. Полезность групп проверяется абляциями выше, где кандидаты фиксированы."))


## 9. Анализ активности и ошибок

Качество измеряется отдельно в трёх группах пользователей, сформированных только по числу событий в истории до начала тестового периода. Это позволяет увидеть, как персонализация меняется с доступным объёмом истории. В таблице пропущенных целей различаются два случая: товар не был найден генераторами либо попал в набор кандидатов, но не вошёл в топ-10.

**Гипотеза.** Качество зависит от объёма истории; ошибки возникают и при генерации, и при ранжировании. Сравниваем группы описательно, без подбора по test.

In [ ]:
activity = (
    test_features.groupby("user_id", as_index=False)["user_total_events"]
    .first()
)
activity["activity_group"] = pd.qcut(
    activity["user_total_events"].rank(method="first"),
    q=3,
    labels=["Низкая", "Средняя", "Высокая"],
)
activity_rows = []
for model_name, per_user in test_per_user.items():
    joined = per_user.merge(activity, on="user_id", how="left", validate="one_to_one")
    for group_name, group in joined.groupby("activity_group", observed=False):
        activity_rows.append({
            "model": model_name,
            "activity_group": str(group_name),
            "ndcg@10": group["ndcg@10"].mean(),
            "users": len(group),
        })
activity_results = pd.DataFrame(activity_rows)
activity_results.to_csv(METRICS / "ltr_activity_groups.csv", index=False)
display(activity_results.pivot(index="activity_group", columns="model",
                               values="ndcg@10").rename_axis(
                                   index="Группа активности", columns="Модель"
                               ).round(4))
fig, ax = plt.subplots(figsize=(10, 5))
activity_results.pivot(index="activity_group", columns="model",
                       values="ndcg@10").plot(kind="bar", ax=ax)
ax.set(title="NDCG@10 на тесте по активности до тестового периода",
       xlabel="Группа исторической активности", ylabel="Средняя NDCG@10")
ax.tick_params(axis="x", labelrotation=0)
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Модель", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "ltr_activity_groups.png", dpi=300,
            bbox_inches="tight")
plt.show()
plt.close("all")

history_seen = (
    pretest_history.groupby("visitorid")["itemid"]
    .agg(lambda values: set(int(value) for value in values))
    .to_dict()
)
candidate_sources = test_candidates.set_index(["user_id", "item_id"])
ranker_top10 = test_recommendations["CatBoostRanker (ALS + ItemKNN)"]
target_pairs = test_benchmark.groupby(["visitorid", "itemid"])["event"].agg(
    lambda values: ", ".join(sorted(set(values)))
)
missed_rows = []
for (user_id, item_id), event_types in target_pairs.items():
    event_types = ", ".join({
        "view": "просмотр", "addtocart": "добавление в корзину",
        "transaction": "покупка",
    }.get(event, event) for event in event_types.split(", "))
    user_id, item_id = int(user_id), int(item_id)
    if item_id in history_seen.get(user_id, set()):
        continue
    if item_id in ranker_top10.get(user_id, []):
        continue
    generated = (user_id, item_id) in candidate_sources.index
    source = candidate_sources.loc[(user_id, item_id), "candidate_source"] if generated else "not generated"
    source = {
        "both": "Обе модели", "als_only": "Только ALS",
        "itemknn_only": "Только ItemKNN", "not generated": "Не сгенерирован",
    }[source]
    missed_rows.append({
        "ID пользователя": user_id,
        "ID целевого товара": item_id,
        "Будущие события": event_types,
        "Кандидат сгенерирован": generated,
        "Источник кандидата": source,
        "Попал в Top-10 ранжировщика": False,
    })
missed_targets = pd.DataFrame(missed_rows)
print("Ненаблюдавшиеся ранее целевые пары, пропущенные CatBoost Top-10:", len(missed_targets))
display(missed_targets.head(10))

In [ ]:
group_scores = activity_results.pivot(index="activity_group", columns="model", values="ndcg@10")
for group in ["Низкая", "Средняя", "Высокая"]:
    cb_score = group_scores.loc[group, "CatBoostRanker (ALS + ItemKNN)"]
    knn = group_scores.loc[group, "ItemKNN"]
    display(Markdown(f"**Активность {group}:** CatBoost={cb_score:.5f}, ItemKNN={knn:.5f}; Δ={cb_score-knn:+.5f}."))
display(Markdown("**Вывод.** Выигрыш среднего нельзя переносить на каждую группу: оцениваем знак и величину каждой разницы выше. Размеры групп приведены в CSV; малые группы и разбиение одинаковой активности по рангам ограничивают вывод. Таблица пропусков иллюстрирует ошибки первого и второго этапов, а их масштаб описывает candidate recall."))


## 10. Выводы и ограничения

Ни одна конфигурация не выбирается по тестовой выборке: параметры ItemKNN и implicit ALS выбраны в ноутбуке 02, CatBoostRanker — по проверочной выборке в этом ноутбуке. Итоговые тестовые метрики, разрыв качества между группами активности, результаты абляций и важность признаков рассчитаны ниже из данных и сохранены в `outputs/metrics/`. Абляции являются диагностикой на проверочной выборке; тест не используется для выбора признаков.

Ранжировщик может переставить только уже сгенерированные товары; недостаточная полнота кандидатов ограничивает его потолок. Бенчмарк охватывает пользователей с историей и ограниченный каталог выбранного режима, определённый по обучающей выборке, поэтому не решает холодный старт.

In [ ]:
ranker.save_model(str(MODELS / "catboost_ranker.cbm"))
experiment_config = {
    "catboost_version": catboost.__version__,
    "benchmark": manifest,
    "relevance_mode": RELEVANCE_MODE,
    "ranker_params": ranker_params,
    "selected_base_params": selected_parameters,
    "catboost_tuning_validation": catboost_tuning.to_dict(orient="records"),
    "feature_ablation_validation": ablation_results.to_dict(orient="records"),
    "selected_catboost_configuration": best_catboost_name,
    "selected_catboost_validation_ndcg@10": float(catboost_tuning.loc[best_catboost_index, "ndcg@10"]),
    "best_iteration": int(best_iteration),
    "early_stopping_rounds": 50,
    "random_seed": SEED,
    "candidate_generators": ["implicit ALS", "ItemKNN"],
    "candidates_per_generator": CANDIDATES_PER_MODEL,
    "rank_training_snapshot_start": rank_snapshot_start,
    "rank_training_snapshot_cutoff": str(rank_cutoff),
    "rank_training_target_end": rank_snapshot_target_end,
    "validation_start": str(validation.event_time.min()),
    "test_start": str(test.event_time.min()),
    "benchmark_users": len(cohort),
    "rank_training_users": len(rank_users),
    "catalog_items": len(catalog),
    "relevance": "binary future user-item interaction",
}
with (MODELS / "catboost_ranker_config.json").open("w") as file:
    json.dump(experiment_config, file, indent=2)

best_validation_name = validation_results["ndcg@10"].idxmax()
print("Лучшая модель по NDCG@10 на проверочной выборке:", best_validation_name)
print("NDCG@10 на тестовой выборке:", test_results["ndcg@10"].to_dict())
print("Таблицы подбора и абляции основаны только на проверочной выборке.")
print("Модель сохранена:", MODELS / "catboost_ranker.cbm")
print("Конфигурация сохранена:", MODELS / "catboost_ranker_config.json")